In [29]:
import torch
from torch import nn, tensor
import torch.nn.functional as F

### Softmax

In [2]:
x = torch.tensor([1.0, 2.0, 3.0])
probs = F.softmax(x, dim=0)
probs

tensor([0.0900, 0.2447, 0.6652])

In [3]:
probs.sum()

tensor(1.)

In [4]:
## Manual
x = torch.tensor([1.0, 2.0, 3.0])
exp_x = torch.exp(x)
exp_x

tensor([ 2.7183,  7.3891, 20.0855])

In [ ]:
# torch.exp(torch.tensor(0))

In [5]:
sum_exp = exp_x.sum()
sum_exp

tensor(30.1929)

In [6]:
softmax = exp_x / sum_exp
softmax

tensor([0.0900, 0.2447, 0.6652])

### Log Softmax

In [7]:
x = torch.tensor([1., 2., 3.])
log_probs = F.log_softmax(x, dim=0)
log_probs

tensor([-2.4076, -1.4076, -0.4076])

In [8]:
torch.log(F.softmax(x, dim=0))

tensor([-2.4076, -1.4076, -0.4076])

### Grad

In [49]:
model = nn.Sequential(
    nn.Linear(1, 3, bias=False)
)
state_dict = {
    "0.weight": torch.tensor([
        [1.],
        [2.],
        [3.]
    ])
}
model.load_state_dict(state_dict)

x = tensor([[2.]])
y = model(x)
print("y:", y)

loss_fn = nn.CrossEntropyLoss()
target = tensor([[0.,0.,1.]])
loss = loss_fn(y, target)
print("loss:", loss)

print("loss-manual:", -1 * F.log_softmax(y, dim=1)[0][2] )
loss.backward()
print("grad:", model[0].weight.grad)

y: tensor([[2., 4., 6.]], grad_fn=<MmBackward0>)
loss: tensor(0.1429, grad_fn=<DivBackward1>)
loss-manual: tensor(0.1429, grad_fn=<MulBackward0>)
grad: tensor([[ 0.0318],
        [ 0.2346],
        [-0.2664]])


In [59]:
log_probs = F.log_softmax(tensor([2., 4., 6.]), dim=0)
log_probs

tensor([-4.1429, -2.1429, -0.1429])

In [61]:
L = (-1 * tensor([0, 0, 1]) * log_probs).sum()
L

tensor(0.1429)

In [63]:
x = tensor(2.)
w0 = tensor(1.)
w1 = tensor(2.)
w2 = tensor(3.)

y0 = x * w0
y1 = x * w1
y2 = x * w2

e_y0 = torch.exp(y0)
e_y1 = torch.exp(y1)
e_y2 = torch.exp(y2)

p0 = e_y0 / (e_y0 + e_y1 + e_y2)
p1 = e_y1 / (e_y0 + e_y1 + e_y2)
p2 = e_y2 / (e_y0 + e_y1 + e_y2)

log_p0 = torch.log(p0)
log_p1 = torch.log(p1)
log_p2 = torch.log(p2)

In [64]:
L = -1 * log_p2
L

tensor(0.1429)

In [65]:
L = -1 * torch.log( e_y2 / (e_y0 + e_y1 + e_y2)  )
L

tensor(0.1429)

In [66]:
# log(a/b) = log a - log b
L = -1 * ( torch.log(e_y2) - torch.log(e_y0 + e_y1 + e_y2)  )
L

tensor(0.1429)

In [68]:
L = -1 * (y2 - torch.log(e_y0 + e_y1 + e_y2))
L

tensor(0.1429)

In [69]:
dL_dy0 = (1/(e_y0 + e_y1 + e_y2)) * e_y0
dL_dy0

tensor(0.0159)

In [70]:
dy0_dw0 = x
dy1_dw1 = x
dy2_dw2 = x

In [71]:
dL_dw0 = dL_dy0 * dy0_dw0
dL_dw0

tensor(0.0318)

In [73]:
L = -1 * (y2 - torch.log(e_y0 + e_y1 + e_y2))
L = torch.log(e_y0 + e_y1 + e_y2) - y2

dL_dy1 = (1/(e_y0 + e_y1 + e_y2)) * e_y1
dL_dy1 * dy1_dw1

tensor(0.2346)

In [75]:
L = -1 * (y2 - torch.log(e_y0 + e_y1 + e_y2))
L = torch.log(e_y0 + e_y1 + e_y2) - y2

dL_dy2 = (1/(e_y0 + e_y1 + e_y2)) * e_y2 - 1
dL_dy2 * dy2_dw2

tensor(-0.2664)